In [1]:
# Cell 1: Setup and connect to database
import pandas as pd
import psycopg2
from dotenv import load_dotenv
import os

load_dotenv(r"C:\Users\Alex\Documents\dissertation-ai\safeguarding_platform\.env")
DATABASE_URL = os.environ.get("DATABASE_URL")

conn = psycopg2.connect(DATABASE_URL)

# Load all reviewed reports where caseworker has made a decision
reviewed_df = pd.read_sql_query("""
    SELECT id, free_text, 
           model_urgency, model_category,
           caseworker_urgency, caseworker_category,
           model_urgency_confidence, model_category_confidence
    FROM reports 
    WHERE caseworker_urgency IS NOT NULL 
    AND caseworker_category IS NOT NULL
""", conn)
conn.close()

print(f"Total reviewed reports: {len(reviewed_df)}")
if len(reviewed_df) > 0:
    print(f"\nSample reviewed report:")
    print(reviewed_df.iloc[0].to_string())

Total reviewed reports: 6

Sample reviewed report:
id                                                                          18
free_text                    It was reported to me over the telephone from ...
model_urgency                                                             High
model_category                                                 Physical safety
caseworker_urgency                                                      Medium
caseworker_category                                  Mental health / self-harm
model_urgency_confidence                                              0.666728
model_category_confidence                                             0.331527


C:\Users\Alex\AppData\Local\Temp\ipykernel_37432\959224726.py:13: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  reviewed_df = pd.read_sql_query("""


In [2]:
# Cell 2: Analyse model vs caseworker agreement
print("=== MODEL vs CASEWORKER AGREEMENT ===\n")

# Urgency agreement
urg_agree = (reviewed_df["model_urgency"] == reviewed_df["caseworker_urgency"]).sum()
urg_total = len(reviewed_df)
print(f"Urgency agreement: {urg_agree}/{urg_total} ({urg_agree/urg_total:.0%})")

# Category agreement
cat_agree = (reviewed_df["model_category"] == reviewed_df["caseworker_category"]).sum()
print(f"Category agreement: {cat_agree}/{urg_total} ({cat_agree/urg_total:.0%})")

# Show disagreements in detail
print("\n=== DISAGREEMENTS ===\n")

urg_disagree = reviewed_df[reviewed_df["model_urgency"] != reviewed_df["caseworker_urgency"]]
if len(urg_disagree) > 0:
    print(f"Urgency disagreements ({len(urg_disagree)}):")
    for _, row in urg_disagree.iterrows():
        print(f"  Report #{row['id']}: Model={row['model_urgency']} ({row['model_urgency_confidence']:.0%}) -> Caseworker={row['caseworker_urgency']}")
        print(f"    Text: {row['free_text'][:120]}...")
        print()

cat_disagree = reviewed_df[reviewed_df["model_category"] != reviewed_df["caseworker_category"]]
if len(cat_disagree) > 0:
    print(f"Category disagreements ({len(cat_disagree)}):")
    for _, row in cat_disagree.iterrows():
        print(f"  Report #{row['id']}: Model={row['model_category']} ({row['model_category_confidence']:.0%}) -> Caseworker={row['caseworker_category']}")
        print(f"    Text: {row['free_text'][:120]}...")
        print()

=== MODEL vs CASEWORKER AGREEMENT ===

Urgency agreement: 2/6 (33%)
Category agreement: 2/6 (33%)

=== DISAGREEMENTS ===

Urgency disagreements (4):
  Report #18: Model=High (67%) -> Caseworker=Medium
    Text: It was reported to me over the telephone from the sister of Cadet John Smith that he had been self harming. He was rushe...

  Report #21: Model=High (99%) -> Caseworker=Medium
    Text: Marie brown has been calling Kristina senior on what's app and telling her she needs to kill herself because she is fat....

  Report #19: Model=Medium (92%) -> Caseworker=Low
    Text: My friend, Jimbo Jones, sent me a snapchat message last night saying that he thinks he might be non-binary and that it's...

  Report #20: Model=High (57%) -> Caseworker=Low
    Text: It was reported to me by a parent that his daughter, Cadet Sgt Sarah Williams, had been getting lots of messages and pho...

Category disagreements (4):
  Report #18: Model=Physical safety (33%) -> Caseworker=Mental health / self-ha

In [3]:
# Cell 3: Generate corrected training examples from caseworker feedback
print("=== GENERATING CORRECTED TRAINING DATA ===\n")

# The caseworker's judgement becomes the authoritative label
# These corrections can be added to the training set for the next model iteration

corrected_records = []
for _, row in reviewed_df.iterrows():
    corrected_records.append({
        "free_text": row["free_text"],
        "urgency_label": row["caseworker_urgency"],
        "category_label": row["caseworker_category"],
        "source": "caseworker_correction",
        "original_model_urgency": row["model_urgency"],
        "original_model_category": row["model_category"],
    })

corrected_df = pd.DataFrame(corrected_records)

print(f"Corrected training examples: {len(corrected_df)}")
print(f"\nUrgency distribution (caseworker-assigned):")
print(corrected_df["urgency_label"].value_counts().sort_index())
print(f"\nCategory distribution (caseworker-assigned):")
print(corrected_df["category_label"].value_counts().sort_index())

# Show how these would be merged with existing training data
existing_train = pd.read_csv(
    r"C:\Users\Alex\Documents\dissertation-ai\safeguarding_platform\data\processed\safeguarding_reports_v4.csv"
)
print(f"\nExisting training records: {len(existing_train)}")
print(f"Corrected records to add: {len(corrected_df)}")
print(f"Combined total: {len(existing_train) + len(corrected_df)}")
print(f"\nThese caseworker-corrected examples carry particular value because")
print(f"they represent cases where the model was wrong and a domain expert")
print(f"provided the correct label. Over time, as more reports are reviewed,")
print(f"the correction set grows and the model progressively improves.")

=== GENERATING CORRECTED TRAINING DATA ===

Corrected training examples: 6

Urgency distribution (caseworker-assigned):
urgency_label
High      1
Low       2
Medium    3
Name: count, dtype: int64

Category distribution (caseworker-assigned):
category_label
Abuse by adult in organisation    1
Abuse by another young person     1
Behaviour / conduct               1
Bullying / peer conflict          1
Mental health / self-harm         2
Name: count, dtype: int64

Existing training records: 3500
Corrected records to add: 6
Combined total: 3506

These caseworker-corrected examples carry particular value because
they represent cases where the model was wrong and a domain expert
provided the correct label. Over time, as more reports are reviewed,
the correction set grows and the model progressively improves.


In [4]:
# Cell 4: Demonstrate the retraining pipeline
print("=== FEEDBACK LOOP RETRAINING DEMONSTRATION ===\n")

# Step 1: Merge corrected examples with existing training data
combined_df = pd.concat([
    existing_train[["free_text", "urgency_label", "category_label"]],
    corrected_df[["free_text", "urgency_label", "category_label"]],
], ignore_index=True)

print(f"Step 1: Merged training data")
print(f"  Synthetic examples: {len(existing_train)}")
print(f"  Caseworker corrections: {len(corrected_df)}")
print(f"  Combined: {len(combined_df)}")

# Step 2: Save the combined dataset for future retraining
output_path = r"C:\Users\Alex\Documents\dissertation-ai\safeguarding_platform\data\processed\training_with_feedback.csv"
combined_df.to_csv(output_path, index=False)
print(f"\nStep 2: Saved combined dataset to {output_path}")

# Step 3: Show the impact over time
print(f"\nStep 3: Projected impact of feedback loop")
print(f"  Current model trained on: 3,500 synthetic examples")
print(f"  Caseworker corrections available: {len(corrected_df)}")
print(f"  Correction rate: {len(corrected_df)}/{len(reviewed_df)} reviews contained a correction")
print()

# Calculate correction patterns
urg_corrections = len(reviewed_df[reviewed_df["model_urgency"] != reviewed_df["caseworker_urgency"]])
cat_corrections = len(reviewed_df[reviewed_df["model_category"] != reviewed_df["caseworker_category"]])

print(f"  Urgency corrections: {urg_corrections}/{len(reviewed_df)} ({urg_corrections/len(reviewed_df):.0%})")
print(f"  Category corrections: {cat_corrections}/{len(reviewed_df)} ({cat_corrections/len(reviewed_df):.0%})")

print(f"""
Step 4: How the feedback loop would operate in production

  1. Reports are submitted and classified by the AI model
  2. Caseworkers review reports and correct urgency/category where needed
  3. Corrections are stored in the database alongside the original predictions
  4. Periodically (e.g. monthly), a retraining pipeline:
     a) Extracts all caseworker-corrected examples from the database
     b) Merges them with the existing training dataset
     c) Retrains the multi-task model on the combined data
     d) Evaluates the new model against a held-out validation set
     e) If performance improves, deploys the updated model
  5. Over time, the model learns from expert corrections and improves
     on exactly the types of cases it previously got wrong

This creates a virtuous cycle: the more the system is used and
corrected, the better it becomes. Importantly, the caseworker
remains the authoritative decision-maker throughout — the AI
provides a starting point that improves progressively.

Current evidence from {len(reviewed_df)} reviewed reports shows that
caseworkers corrected the model's urgency in {urg_corrections} cases
and category in {cat_corrections} cases. Each correction represents
a high-value training example because it captures expert judgement
on a case the model found difficult.
""")

=== FEEDBACK LOOP RETRAINING DEMONSTRATION ===

Step 1: Merged training data
  Synthetic examples: 3500
  Caseworker corrections: 6
  Combined: 3506

Step 2: Saved combined dataset to C:\Users\Alex\Documents\dissertation-ai\safeguarding_platform\data\processed\training_with_feedback.csv

Step 3: Projected impact of feedback loop
  Current model trained on: 3,500 synthetic examples
  Caseworker corrections available: 6
  Correction rate: 6/6 reviews contained a correction

  Urgency corrections: 4/6 (67%)
  Category corrections: 4/6 (67%)

Step 4: How the feedback loop would operate in production

  1. Reports are submitted and classified by the AI model
  2. Caseworkers review reports and correct urgency/category where needed
  3. Corrections are stored in the database alongside the original predictions
  4. Periodically (e.g. monthly), a retraining pipeline:
     a) Extracts all caseworker-corrected examples from the database
     b) Merges them with the existing training dataset
    